<!--nav--> [🗺 Learning path](README.md) · **28/39** · ◀ [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb) · [Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb) ▶

# Structured Output & Guided Decoding: Making JSON Impossible to Get Wrong

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Structured_Output_Guided_Decoding.ipynb)

Every agent, every tool call, every "extract these fields from this document" pipeline needs the
model to emit **valid JSON matching a schema**. The naive approach — ask nicely, parse, retry on
failure — is the single most expensive habit in production LLM systems.

There's a better way, and it's a *serving* feature: **constrained decoding**. Instead of hoping for
valid JSON, the engine makes invalid JSON **unrepresentable** by masking illegal tokens at every
step. Validity goes to 100% by construction, and modern implementations cost almost nothing.

| Part | What you'll learn |
|---|---|
| **1** | The retry tax: what "ask nicely and retry" actually costs |
| **2** | **The mechanism**, built from scratch — a token-masking FSM you can read in 40 lines |
| **3** | An interactive D3 visualization: watch the mask change token by token |
| **4** | Real backends: **xgrammar**, Outlines, llguidance — and why the overhead is now ~free |
| **5** | Using it in vLLM (`guided_json`, `guided_regex`, `guided_grammar`, tool calling) |
| **6** | The traps: schema shape hurts quality, grammar compile cost, what constraints *can't* fix |

**Runs on:** any CPU. Parts 1–3 are pure Python. Part 5's live section is GPU-gated.

In [ ]:
import json, re, math, random, uuid, statistics
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · The retry tax

The standard pipeline looks like this:

```
prompt ──► model ──► text ──► json.loads() ──► ✅ done
                                  │
                                  └─ ❌ ValueError ──► retry (new prompt, new tokens, new latency)
```

Every retry costs a **full request**: prefill + decode + queue time. And retries don't just cost
money — they cost *tail latency*, which is what your SLO measures (notebook 27). A 10% failure rate
doesn't make p95 10% worse; it can double it, because the retried request goes to the *back* of the
queue.

Let's price it. Assume a per-attempt validity rate `p` (real numbers for a small model asked for
nested JSON without constraints are often 0.7–0.95):

In [ ]:
def retry_cost(p_valid, max_attempts=4, tokens_per_attempt=180, ttft=0.4, tpot=0.03):
    '''Expected tokens & latency under retry-until-valid, plus the failure rate that leaks to users.'''
    exp_attempts = sum(k * (p_valid * (1 - p_valid) ** (k - 1)) for k in range(1, max_attempts + 1))
    leak = (1 - p_valid) ** max_attempts                       # still broken after all retries
    exp_attempts += max_attempts * leak                        # failures burn every attempt
    per_attempt_latency = ttft + tpot * tokens_per_attempt
    return {"exp_attempts": exp_attempts,
            "exp_tokens": exp_attempts * tokens_per_attempt,
            "exp_latency": exp_attempts * per_attempt_latency,
            "unfixed_failure_rate": leak}

print(f"{'validity/attempt':>17}{'attempts':>10}{'tokens':>9}{'latency':>10}{'still broken':>14}")
print("-" * 61)
for p in (0.70, 0.85, 0.95, 0.99, 1.00):
    r = retry_cost(p)
    print(f"{p:>17.0%}{r['exp_attempts']:>10.2f}{r['exp_tokens']:>9.0f}"
          f"{r['exp_latency']:>9.2f}s{r['unfixed_failure_rate']:>13.2%}")

base = retry_cost(1.0)
worst = retry_cost(0.70)
print(f"\nAt 70% per-attempt validity you pay {worst['exp_tokens']/base['exp_tokens']:.2f}x the tokens")
print(f"and {worst['exp_latency']/base['exp_latency']:.2f}x the latency - AND 0.81% of users still get")
print("garbage after 4 tries. Constrained decoding makes that column exactly 0.00%.")

## Part 2 · The mechanism, from scratch

Here's the whole idea in one sentence: **at each decoding step, set the logits of every token that
would make the output invalid to −∞ before sampling.**

```
       logits over the full vocabulary            mask from the grammar's current state
   [ 2.1, -0.4, 5.5, 0.9, 3.2, ... ]        ×    [ 1,   0,    1,   0,   1, ... ]
                                                  ↓
                        [ 2.1, -inf, 5.5, -inf, 3.2, ... ]  ──► softmax ──► sample
                                                  ↓
                          the sampled token is GUARANTEED to be grammatical
```

The only hard part is computing that mask fast. The grammar is compiled to a **state machine**;
each state knows which tokens may come next. Let's build a real (if tiny) one for a JSON object
with a fixed schema, so you can see every moving part.

We'll use a character-level "vocabulary" to keep it readable — real engines do the same thing over
BPE tokens, which is fiddlier only because one token can span several grammar characters.

In [ ]:
VOCAB = list('{}",:[]abcdefghijklmnopqrstuvwxyz0123456789_. -') + ["<eos>"]

class SchemaFSM:
    '''Enforces: {"name": "<string>", "age": <int>, "active": <bool>}  (in that order).

    Each state returns the SET of characters legal as the very next character.
    That set is exactly the mask a serving engine applies to the logits.
    '''
    def __init__(self):
        self.reset()

    def reset(self):
        self.buf = ""
        self.state = "start"
        self.field = 0
        return self

    FIELDS = [("name", "string"), ("age", "int"), ("active", "bool")]

    def allowed(self):
        s, f = self.state, self.field
        if s == "start":        return {"{"}
        if s == "pre_key":      return {'"'}
        if s == "in_key":                                   # only the exact expected key
            key = self.FIELDS[f][0]
            done = self.buf
            return {key[len(done)]} if len(done) < len(key) else {'"'}
        if s == "post_key":     return {":"}
        if s == "pre_val":
            kind = self.FIELDS[f][1]
            if kind == "string": return {'"'}
            if kind == "int":    return set("123456789")     # no leading zero
            if kind == "bool":   return {"t", "f"}
        if s == "in_string":
            # maxLength=24, exactly like a JSON-schema bound: past it, only the closing quote is legal
            return {'"'} if len(self.buf) >= 24 else set("abcdefghijklmnopqrstuvwxyz _-") | {'"'}
        if s == "in_int":
            close = {","} if f < 2 else {"}"}
            return close if len(self.buf) >= 5 else set("0123456789") | close
        if s == "in_bool":
            word = "true" if self.buf[0] == "t" else "false"
            return {word[len(self.buf)]} if len(self.buf) < len(word) else ({","} if f < 2 else {"}"})
        if s == "post_val":     return {","} if f < len(self.FIELDS) - 1 else {"}"}
        if s == "done":         return {"<eos>"}
        return set()

    def step(self, ch):
        s = self.state
        if s == "start" and ch == "{":       self.state, self.buf = "pre_key", ""
        elif s == "pre_key" and ch == '"':   self.state, self.buf = "in_key", ""
        elif s == "in_key":
            if ch == '"':                    self.state = "post_key"
            else:                            self.buf += ch
        elif s == "post_key" and ch == ":":  self.state, self.buf = "pre_val", ""
        elif s == "pre_val":
            kind = self.FIELDS[self.field][1]
            if kind == "string" and ch == '"': self.state, self.buf = "in_string", ""
            elif kind == "int":                self.state, self.buf = "in_int", ch
            elif kind == "bool":               self.state, self.buf = "in_bool", ch
        elif s == "in_string":
            if ch == '"':                    self.state = "post_val"
            else:                            self.buf += ch
        elif s == "in_int":
            if ch in ",}":                   self.state = "post_val"; return self.step(ch)
            else:                            self.buf += ch
        elif s == "in_bool":
            word = "true" if self.buf[0] == "t" else "false"
            if len(self.buf) < len(word):    self.buf += ch
            if self.buf == word and ch in ",}": self.state = "post_val"; return self.step(ch)
        elif s == "post_val":
            if ch == ",":                    self.state = "pre_key"; self.field += 1
            elif ch == "}":                  self.state = "done"
        return self

# Drive it with a model that outputs PURE NOISE - if the FSM works, the output is still valid JSON.
def generate_masked(fsm, rng, max_chars=120):
    fsm.reset(); out = []
    for _ in range(max_chars):
        allowed = fsm.allowed()
        if allowed == {"<eos>"}: break
        ch = rng.choice(sorted(allowed))        # a random pick among LEGAL characters only
        out.append(ch); fsm.step(ch)
    return "".join(out)

print("Three samples from a *uniformly random* generator, constrained by the FSM:\n")
for i in range(3):
    text = generate_masked(SchemaFSM(), random.Random(i))
    try:
        obj = json.loads(text)
        print(f"  ✓ {text}\n      json.loads -> {obj}")
    except Exception as e:
        print(f"  ✗ {text}\n      {e}")

**Read that again: the "model" was a random number generator** — the worst possible language
model — and every output still parsed as schema-correct JSON. That's the guarantee constrained
decoding gives you. Quality of *content* still depends on the model; **validity of *form* no longer
does.**

Now compare against the unconstrained baseline, where the same random generator picks from the whole
vocabulary:

In [ ]:
def generate_unmasked(rng, max_chars=120):
    return "".join(rng.choice(VOCAB[:-1]) for _ in range(rng.randint(20, 60)))

trials = 2000
valid_masked = valid_free = 0
for i in range(trials):
    t1 = generate_masked(SchemaFSM(), random.Random(i))
    try: json.loads(t1); valid_masked += 1
    except Exception: pass
    t2 = generate_unmasked(random.Random(i))
    try: json.loads(t2); valid_free += 1
    except Exception: pass

print(f"constrained : {valid_masked}/{trials} valid  ({valid_masked/trials:.1%})")
print(f"unconstrained: {valid_free}/{trials} valid  ({valid_free/trials:.1%})")
print("\nA real model is obviously far better than random - but the SHAPE of this result is the point:")
print("constraint gives a 100% floor that no amount of prompt engineering can guarantee.")

## Part 3 · Watch the mask move

Step through a generation and see, at each character, which parts of the vocabulary are legal
(green) and which are masked to −∞ (grey). This is literally what the engine does to the logit
vector before sampling.

In [ ]:
# Record the (state, allowed-set) trace of one constrained generation for the visualization.
def trace(fsm, rng, max_chars=90):
    fsm.reset(); steps = []
    for _ in range(max_chars):
        allowed = sorted(fsm.allowed())
        if allowed == ["<eos>"]:
            steps.append({"state": fsm.state, "allowed": allowed, "chosen": "<eos>",
                          "so_far": "".join(s["chosen"] for s in steps)})
            break
        ch = rng.choice(allowed)
        steps.append({"state": fsm.state, "allowed": allowed, "chosen": ch,
                      "so_far": "".join(s["chosen"] for s in steps)})
        fsm.step(ch)
    return steps

tr = trace(SchemaFSM(), random.Random(3))
print(f"{len(tr)} decoding steps; final text: {''.join(s['chosen'] for s in tr[:-1])}")
print(f"\nAt step 0 the mask allows {tr[0]['allowed']} - literally one legal token out of {len(VOCAB)}.")
avg_allowed = statistics.mean(len(s['allowed']) for s in tr)
print(f"Average legal tokens per step: {avg_allowed:.1f} / {len(VOCAB)} "
      f"({avg_allowed/len(VOCAB):.0%} of the vocabulary)")

In [ ]:
JS = r'''
const vocab = data.vocab, steps = data.steps;
const cell = 26, perRow = Math.floor((W - 20) / cell);

const bar = root.append("div").style("margin-bottom","6px");
const btn = bar.append("button").text("▶ play");
const scrub = bar.append("input").attr("type","range").attr("min",0).attr("max",steps.length-1)
    .attr("value",0).style("width","280px").style("margin-left","10px").style("vertical-align","middle");
const out = root.append("div").style("font","15px ui-monospace,monospace").style("background","#f6f8fa")
    .style("padding","8px").style("border-radius","6px").style("min-height","24px").style("margin-bottom","8px");
const info = root.append("div").style("font","12.5px system-ui").style("margin-bottom","6px");

const svg = root.append("svg").attr("width", W).attr("height", Math.ceil(vocab.length/perRow)*cell + 30);
const g = svg.append("g").attr("transform","translate(6,18)");
const rects = g.selectAll("r").data(vocab).join("rect")
    .attr("x",(d,i)=>(i%perRow)*cell).attr("y",(d,i)=>Math.floor(i/perRow)*cell)
    .attr("width",cell-3).attr("height",cell-3).attr("rx",3);
const texts = g.selectAll("t").data(vocab).join("text")
    .attr("x",(d,i)=>(i%perRow)*cell + (cell-3)/2).attr("y",(d,i)=>Math.floor(i/perRow)*cell + 15)
    .attr("text-anchor","middle").style("font","11px ui-monospace,monospace")
    .text(d => d === "<eos>" ? "eos" : d);

function render(i) {
  const s = steps[i];
  scrub.property("value", i);
  const allowed = new Set(s.allowed);
  rects.attr("fill", d => d === s.chosen ? "#1b5e20" : allowed.has(d) ? "#a5d6a7" : "#eceff1")
       .attr("stroke", d => d === s.chosen ? "#1b5e20" : "none").attr("stroke-width",2);
  texts.attr("fill", d => d === s.chosen ? "white" : allowed.has(d) ? "#1b5e20" : "#b0bec5");
  out.text(s.so_far + "▌");
  info.html(`step <b>${i}</b> · grammar state <b>${s.state}</b> · ` +
            `<span style="color:#1b5e20">${s.allowed.length} legal</span> / ` +
            `<span style="color:#78909c">${vocab.length - s.allowed.length} masked to −∞</span> · ` +
            `sampled <b>${s.chosen === " " ? "␣" : s.chosen}</b>`);
}
let i = 0, timer = null;
btn.on("click", () => {
  if (timer) { timer.stop(); timer=null; btn.text("▶ play"); }
  else { timer = d3.interval(()=>{ i=(i+1)%steps.length; render(i); }, 260); btn.text("⏸ pause"); }
});
scrub.on("input", function(){ i=+this.value; render(i); });
render(0);
'''
show_d3(JS, {"vocab": VOCAB, "steps": tr}, height=340)

**What to notice.**
- At step 0 exactly **one** token is legal (`{`). The model's opinion is irrelevant — and that's fine,
  because there's only one correct choice.
- While spelling a **key**, the mask allows exactly one character at a time. Keys cost tokens but zero
  decisions. (Real engines exploit this with *jump-forward decoding*: if only one continuation is
  possible, emit it **without a forward pass at all** — free tokens.)
- Inside a **string value** the mask opens up: this is where the model's actual intelligence goes.
- The constraint is *structural*, never semantic: it guarantees `"age": 42`, not that 42 is right.

## Part 4 · Real backends: why this is now ~free

Naively, computing a mask per step per sequence is expensive: vocabularies are ~150k tokens, and you
need the mask *before* sampling, on the critical path. Three tricks made it cheap:

| Trick | What it does |
|---|---|
| **Precomputed context-independent masks** | most grammar states depend only on the state, not the history → compute once at compile time, reuse forever |
| **Persistent execution stack** | recursive grammars (nested JSON) tracked with a stack that's *copied cheaply* per branch instead of re-derived |
| **Overlapping with GPU compute** | build the next step's mask on the **CPU** while the GPU runs the forward pass — the mask is ready before the logits are |
| **Jump-forward / token healing** | when only one continuation is legal, skip the forward pass entirely and emit it |

| Backend | Notes |
|---|---|
| **xgrammar** | vLLM's default structured-output backend; C++ core, the tricks above; near-zero measured overhead in the common case |
| **Outlines** | the library that popularized regex/JSON-schema→FSM decoding; still widely used |
| **llguidance** | Rust engine used by Guidance; very fast general CFG support |

The practical consequence: **you should default to constrained decoding for any machine-consumed
output.** The old trade-off ("structure costs latency") is largely gone; what remains is the
*compile* cost of a new grammar, which is cached per schema.

## Part 5 · Doing it in vLLM

The offline API takes a `guided_*` field in `SamplingParams` (via `GuidedDecodingParams`); the server
takes the same thing in the request body, and also implements OpenAI's `response_format` and tool
calling on top of it.

```python
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams

schema = {
    "type": "object",
    "properties": {
        "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "keywords": {"type": "array", "items": {"type": "string"}, "maxItems": 5},
    },
    "required": ["sentiment", "confidence", "keywords"],
}

sp = SamplingParams(temperature=0.7, max_tokens=200,
                    guided_decoding=GuidedDecodingParams(json=schema))
out = llm.generate([prompt], sp)          # the text WILL match the schema
```

Over HTTP, with any OpenAI client:

```python
client.chat.completions.create(
    model=MODEL, messages=[...],
    extra_body={"guided_json": schema},              # vLLM-native
    # or the portable form:
    response_format={"type": "json_schema", "json_schema": {"name": "s", "schema": schema}},
)
```

Other constraint types, all backed by the same machinery:

```python
GuidedDecodingParams(regex=r"\d{4}-\d{2}-\d{2}")      # dates, IDs, phone numbers
GuidedDecodingParams(choice=["yes", "no", "unsure"])    # classification — the mask is the label set
GuidedDecodingParams(grammar=sql_ebnf)                  # full context-free grammars (SQL, DSLs)
```

And for **tool calling**, launch the server with a parser so OpenAI-style `tools=[...]` works:

```bash
vllm serve Qwen/Qwen2.5-7B-Instruct \
  --enable-auto-tool-choice --tool-call-parser hermes
```

The live cell below runs the real thing if you have a GPU:

In [ ]:
# GPU-ONLY: measure constrained vs unconstrained validity and the overhead of the constraint.
import torch
if not torch.cuda.is_available():
    print("No GPU - skipping. Parts 1-4 already demonstrated the mechanism end to end.")
else:
    import time
    from vllm import LLM, SamplingParams
    from vllm.sampling_params import GuidedDecodingParams

    MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
    llm = LLM(model=MODEL, dtype="half", max_model_len=2048, gpu_memory_utilization=0.85)
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL)

    SCHEMA = {
        "type": "object",
        "properties": {
            "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
            "confidence": {"type": "number"},
            "keywords": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["sentiment", "confidence", "keywords"],
    }
    REVIEWS = ["The battery lasts forever and the screen is gorgeous.",
               "Arrived broken, support never replied. Avoid.",
               "It's fine. Does the job, nothing special.",
               "Setup took three hours but it works beautifully now.",
               "Cheap plastic, but honestly worth the price.",
               "The update bricked it. Furious.",
               "Exactly as described, fast shipping.",
               "Mediocre sound, great build quality."] * 4        # 32 requests

    def prompt(r):
        return tok.apply_chat_template(
            [{"role": "user",
              "content": f"Analyze this review and reply with JSON containing sentiment, "
                         f"confidence and keywords.\n\nReview: {r}"}],
            add_generation_prompt=True, tokenize=False)

    prompts = [prompt(r) for r in REVIEWS]
    llm.generate([prompts[0]], SamplingParams(max_tokens=8))       # warmup

    def run(label, sp):
        t0 = time.perf_counter()
        outs = llm.generate(prompts, sp)
        dt = time.perf_counter() - t0
        texts = [o.outputs[0].text for o in outs]
        ok = 0
        for t in texts:
            try:
                obj = json.loads(t)
                if all(k in obj for k in SCHEMA["required"]): ok += 1
            except Exception: pass
        ntok = sum(len(o.outputs[0].token_ids) for o in outs)
        print(f"{label:<26}{dt:6.1f}s{ntok/dt:>9.0f} tok/s   schema-valid: {ok}/{len(texts)} ({ok/len(texts):.0%})")
        return texts

    free = run("unconstrained", SamplingParams(temperature=0.7, max_tokens=160))
    guided = run("guided_json (xgrammar)",
                 SamplingParams(temperature=0.7, max_tokens=160,
                                guided_decoding=GuidedDecodingParams(json=SCHEMA)))
    print("\nexample constrained output:", guided[0][:180])
    print("\nThe throughput columns are the real headline: structure is essentially free,")
    print("while validity goes to 100% by construction.")

## Part 6 · The traps (read this before shipping)

**1. Constraints fix form, not truth.** A schema guarantees `"confidence": 0.93` is a number. It
does not make 0.93 *calibrated*. Teams routinely ship a schema and assume the content improved.

**2. Schema shape affects quality.** Forcing the model to emit fields in an awkward order can hurt
accuracy — most notably, putting a conclusion **before** its reasoning denies the model its
scratchpad. Prefer:

```json
{"reasoning": "…", "answer": "…"}      ← good: reasoning first, then commit
{"answer": "…", "reasoning": "…"}      ← worse: forces the answer before thinking
```

**3. Over-constraining creates dead ends.** A `maxItems` that's too small, or an enum missing the
right option, forces the model to pick something wrong. The grammar always wins the argument —
make sure it's arguing for the right thing (include an `"other"`/`null` escape hatch where sensible).

**4. Grammar compilation is cached per schema — but it isn't free the first time.** Generating a
*new* schema per request (e.g. schemas built from user input) defeats the cache. Keep schemas static
and parameterize the prompt instead.

**5. Not every constraint is a grammar.** Cross-field logic ("`end_date` must be after `start_date`")
isn't expressible in a CFG. Validate that in code after decoding — constrained decoding removes
parse errors, not business-rule errors.

**5b. `max_tokens` can still truncate you into invalid JSON.** This is the one that bites people who
believe the "100% valid" headline. The grammar guarantees every *prefix* is on a path to a valid
document — it cannot conjure a closing brace once your token budget runs out mid-string. (Our Part 2
FSM hit exactly this: before we bounded string length to 24 characters, ~8% of samples were cut off
mid-value and failed to parse.) **Fixes:** bound your fields (`maxLength`, `maxItems`) so the worst
case fits, set `max_tokens` with headroom, and always check `finish_reason == "length"` —
notebook 26 showed that `vllm:request_success_total{finished_reason="length"}` is a metric worth
graphing for precisely this reason.

**6. `"JSON mode"` ≠ schema enforcement.** Some APIs' generic "JSON mode" only guarantees *some*
valid JSON, not *your* schema. Always pass the schema.

## Recap

- **Retries are a tax** on tokens *and* tail latency, and they never reach 100% validity.
- **Constrained decoding masks illegal tokens** at each step — you built the mechanism yourself in
  Part 2 and watched it in Part 3.
- Modern backends (**xgrammar** in vLLM) make it near-free via precomputed masks, persistent stacks,
  CPU/GPU overlap, and jump-forward decoding.
- Use it for **everything machine-consumed**: tool calls, extraction, classification, agent steps.
- It guarantees **form, not content** — and schema design is now part of prompt engineering.

### Further reading
- [xgrammar](https://github.com/mlc-ai/xgrammar) · [Outlines](https://github.com/dottxt-ai/outlines) · [llguidance](https://github.com/guidance-ai/llguidance)
- [vLLM structured outputs docs](https://docs.vllm.ai/en/latest/features/structured_outputs.html) · [vLLM tool calling](https://docs.vllm.ai/en/latest/features/tool_calling.html)
- [Efficient Guided Generation for LLMs](https://arxiv.org/abs/2307.09702) (the Outlines FSM paper)
- [SGLang / RadixAttention](https://arxiv.org/abs/2312.07104) — compressed FSMs and jump-forward decoding

▶ **Next:** [Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb) — one GPU
isn't enough; how do you shard and route?